# Training Analysis

Analyze training results from PKS-MPNN experiments:
- Loss curves
- Per-domain performance
- Comparison across experiment types
- wandb integration

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from glob import glob

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## Load wandb Results (if available)

In [ ]:
try:
    import wandb
    api = wandb.Api()
    
    # Fetch runs from PKS-MPNN project
    runs = api.runs('PKS-MPNN')
    
    print(f'Found {len(runs)} wandb runs')
    for run in runs[:10]:
        print(f'  {run.name}: {run.state}')
except Exception as e:
    print(f'wandb not available or not logged in: {e}')
    runs = []

## Load Local Evaluation Results

In [ ]:
# Load evaluation results from different experiments
eval_dir = Path('../outputs/evaluation')

results = {}
if eval_dir.exists():
    for eval_file in eval_dir.glob('evaluation_*.json'):
        with open(eval_file) as f:
            data = json.load(f)
        
        # Extract experiment name from checkpoint path
        checkpoint = Path(data.get('checkpoint', ''))
        exp_name = checkpoint.parent.parent.name if checkpoint.parent else 'unknown'
        
        results[exp_name] = data
    
    print(f'Loaded results for {len(results)} experiments:')
    for name in results:
        print(f'  - {name}')
else:
    print(f'Evaluation directory not found: {eval_dir}')

## Compare Experiment Results

In [ ]:
if results:
    # Create comparison dataframe
    comparison = []
    for exp_name, data in results.items():
        overall = data.get('overall', {})
        comparison.append({
            'Experiment': exp_name,
            'Perplexity': overall.get('perplexity', np.nan),
            'Recovery': overall.get('recovery', np.nan) * 100,
            'NLL': overall.get('loss', np.nan),
            'N Residues': overall.get('n_residues', 0),
        })
    
    df = pd.DataFrame(comparison)
    print('\nOverall Results Comparison:')
    print(df.to_string(index=False))
    
    # Visualize
    if len(df) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        
        # Perplexity
        ax = axes[0]
        ax.bar(df['Experiment'], df['Perplexity'], color='steelblue')
        ax.set_ylabel('Perplexity')
        ax.set_title('Perplexity by Experiment')
        ax.tick_params(axis='x', rotation=45)
        
        # Recovery
        ax = axes[1]
        ax.bar(df['Experiment'], df['Recovery'], color='coral')
        ax.set_ylabel('Recovery (%)')
        ax.set_title('Sequence Recovery by Experiment')
        ax.tick_params(axis='x', rotation=45)
        
        plt.tight_layout()
        plt.show()

## Per-Domain Performance

In [ ]:
if results:
    # Get per-domain results
    domain_data = []
    
    for exp_name, data in results.items():
        per_domain = data.get('per_domain', {})
        for domain, metrics in per_domain.items():
            domain_data.append({
                'Experiment': exp_name,
                'Domain': domain,
                'Perplexity': metrics.get('perplexity', np.nan),
                'Recovery': metrics.get('recovery', np.nan) * 100,
            })
    
    if domain_data:
        df_domain = pd.DataFrame(domain_data)
        
        # Pivot for heatmap
        pivot_ppl = df_domain.pivot(index='Domain', columns='Experiment', values='Perplexity')
        pivot_rec = df_domain.pivot(index='Domain', columns='Experiment', values='Recovery')
        
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        
        # Perplexity heatmap
        ax = axes[0]
        sns.heatmap(pivot_ppl, annot=True, fmt='.2f', cmap='RdYlGn_r', ax=ax)
        ax.set_title('Perplexity by Domain (lower = better)')
        
        # Recovery heatmap
        ax = axes[1]
        sns.heatmap(pivot_rec, annot=True, fmt='.1f', cmap='RdYlGn', ax=ax)
        ax.set_title('Recovery (%) by Domain (higher = better)')
        
        plt.tight_layout()
        plt.show()

## Confidence-Stratified Performance

In [ ]:
if results:
    conf_data = []
    
    for exp_name, data in results.items():
        per_conf = data.get('per_confidence', {})
        for conf_bin, metrics in per_conf.items():
            conf_data.append({
                'Experiment': exp_name,
                'pLDDT Range': conf_bin,
                'Recovery': metrics.get('recovery', np.nan) * 100,
            })
    
    if conf_data:
        df_conf = pd.DataFrame(conf_data)
        
        fig, ax = plt.subplots(figsize=(10, 5))
        
        # Group bar plot
        x = np.arange(len(df_conf['pLDDT Range'].unique()))
        width = 0.8 / len(results)
        
        for i, exp_name in enumerate(results.keys()):
            exp_data = df_conf[df_conf['Experiment'] == exp_name]
            ax.bar(x + i*width, exp_data['Recovery'], width, label=exp_name, alpha=0.8)
        
        ax.set_xlabel('pLDDT Range')
        ax.set_ylabel('Recovery (%)')
        ax.set_title('Sequence Recovery by Confidence Level')
        ax.set_xticks(x + width * len(results) / 2)
        ax.set_xticklabels(df_conf['pLDDT Range'].unique())
        ax.legend()
        
        plt.tight_layout()
        plt.show()

## Summary

Key takeaways from training analysis:

1. **Domain-only**: Expected to have good within-domain recovery but poor interface prediction
2. **Full module**: Maximum context, may struggle with noisy low-confidence regions
3. **Context-aware**: Balanced approach, should show best overall performance

Check wandb for detailed training curves and real-time monitoring.